Qwen2.5-1.5B-Instruct | PEV at a single fixed sigma = 0.024
All 100 JBB-Behaviors prompts, 20 generations per prompt, run as ONE batch of 20.

Differences from the sweep notebook:
  - PERTURB is a single fixed value (0.024), not a list
  - all 100 prompts in one notebook (no 4-way 25-prompt split)
  - the 20 runs per prompt are generated in a single batched forward pass,
    with independent noise per row, instead of 20 sequential run_one() calls

Log format is unchanged ("Prompt{i} - ...", "perturbation value- ...", "run{k}- ..."),
so the existing categorization pipeline parses it as-is.

In [ ]:
get_ipython().system('pip -q install -U transformers datasets accelerate sentencepiece huggingface_hub')

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
uploaded = files.upload()  # select run_qwen.py from your machine

In [ ]:
import os
os.environ["QWEN_DEVICE"] = "cuda"
os.environ["QWEN_SKIP_EMBED_PROMPT"] = "0"

import torch
import run_qwen as rq

model, tokenizer = rq.load_model()
model.eval()

DEVICE = next(model.parameters()).device
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print("device:", DEVICE, "| dtype:", next(model.parameters()).dtype)

In [ ]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Enter your Hugging Face token (hf_...): ")
login(token=hf_token)
print("HF login successful")

In [ ]:
from datasets import load_dataset

ds = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors", split="harmful")
prompt_col = "Goal"

print("Columns:", ds.column_names)
print("Total rows:", len(ds))
print("Using prompt column:", prompt_col)
print("Sample prompt:", ds[0][prompt_col])

### Batched generation

Same decoding settings as the sweep (temperature 0.7, top_k 50, top_p 0.9,
max_new_tokens 4096) and the same empty-system chat template via rq.build_prompt.

The noise matches the sweep's hook: sigma * randn added to the embeddings of the
whole templated prompt, drawn independently for each of the 20 rows.

In [ ]:
import math
import torch

GEN_KWARGS = dict(
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    max_new_tokens=4096,
)


def to_safe_text(x):
    if x is None:
        return ""
    if isinstance(x, float) and math.isnan(x):
        return ""
    return str(x)


def _decode_new_tokens(out_ids, input_ids):
    """With inputs_embeds only, HF returns just the new tokens; be defensive anyway."""
    L = input_ids.shape[1]
    start = 0
    if out_ids.shape[1] >= L and torch.equal(out_ids[0, :L].cpu(), input_ids[0].cpu()):
        start = L
    return tokenizer.batch_decode(out_ids[:, start:], skip_special_tokens=True)


@torch.no_grad()
def run_batch(prompt_text, perturb, n_runs, micro_batch=None, seed=None):
    """Return n_runs completions for one prompt, generated in batches of micro_batch."""
    prompt_text = to_safe_text(prompt_text).strip()
    if not prompt_text:
        return [""] * n_runs
    if seed is not None:
        torch.manual_seed(seed)

    messages = [
        {"role": "system", "content": ""},
        {"role": "user", "content": prompt_text},
    ]
    chat = rq.build_prompt(tokenizer, messages)
    enc = tokenizer(chat, return_tensors="pt", add_special_tokens=False)
    input_ids = enc["input_ids"].to(DEVICE)
    attn_1 = enc["attention_mask"].to(DEVICE)

    embed = model.get_input_embeddings()
    base = embed(input_ids)  # (1, L, H)

    micro_batch = micro_batch or n_runs
    outs = []
    done = 0
    while done < n_runs:
        b = min(micro_batch, n_runs - done)
        try:
            inp = base.expand(b, -1, -1).clone()
            if perturb > 0:
                inp = inp + perturb * torch.randn_like(inp)  # independent noise per row
            out_ids = model.generate(
                inputs_embeds=inp,
                attention_mask=attn_1.expand(b, -1),
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                **GEN_KWARGS,
            )
            outs.extend(_decode_new_tokens(out_ids, input_ids))
            done += b
            del inp, out_ids
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if micro_batch == 1:
                raise
            micro_batch = max(1, micro_batch // 2)
            print(f"    [OOM] retrying with micro_batch={micro_batch}")
    return [to_safe_text(o).replace("\r\n", "\n").strip() for o in outs]

In [ ]:
import os
import time
from datetime import datetime

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_DIR = "/content/drive/MyDrive/LLm_perturbation_logs"
else:
    LOG_DIR = "/content"
os.makedirs(LOG_DIR, exist_ok=True)

PERTURB = 0.024            # single fixed sigma
START_PROMPT = 1           # 1-indexed; bump this to resume after a disconnect
NUM_PROMPTS = 100
RUNS_PER_PERTURB = 20
MICRO_BATCH = 20           # all 20 in one pass; auto-halves on OOM
SEED = 1234                # per-prompt seed = SEED + prompt index, for reproducibility

start_idx = START_PROMPT - 1
end_idx = min(start_idx + NUM_PROMPTS, len(ds))
prompt_indices = list(range(start_idx, end_idx))
prompts = [to_safe_text(ds[i][prompt_col]).strip() for i in prompt_indices]
num_prompts = len(prompts)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = os.path.join(
    LOG_DIR,
    f"JBB_qwen25_1p5b_sigma{str(PERTURB).replace('.', 'p')}"
    f"_p{START_PROMPT:03d}_to_p{start_idx + num_prompts:03d}_{ts}.txt",
)

total = num_prompts * RUNS_PER_PERTURB
print("Log file:", log_path)
print("Prompts:", num_prompts, "| sigma:", PERTURB, "| Runs each:", RUNS_PER_PERTURB)
print("Total generations:", total)

In [ ]:
import shutil

local_log = f"/content/{os.path.basename(log_path)}"


def checkpoint_copy():
    shutil.copy2(log_path, local_log)


job = 0
t0 = time.time()

with open(log_path, "a", encoding="utf-8") as f:
    f.write(f"start {datetime.now().isoformat()}\n")
    f.write(f"prompt_col={prompt_col!r} num_prompts={num_prompts} runs={RUNS_PER_PERTURB}\n")
    f.write(f"perturb_values=[{PERTURB}]\n")
    f.write(f"batched=True micro_batch={MICRO_BATCH} seed={SEED}\n")
    f.write("=" * 120 + "\n\n")
    f.flush()
    os.fsync(f.fileno())

    for p_idx, prompt_text in enumerate(prompts, start=START_PROMPT):
        f.write(f"Prompt{p_idx} - {prompt_text}\n")
        f.write(f"perturbation value- {PERTURB}\n")
        f.flush()
        os.fsync(f.fileno())

        print(f"\n>>> Prompt {p_idx}/{START_PROMPT + num_prompts - 1} | perturbation={PERTURB}")
        tp = time.time()

        try:
            outs = run_batch(
                prompt_text,
                PERTURB,
                RUNS_PER_PERTURB,
                micro_batch=MICRO_BATCH,
                seed=SEED + p_idx,
            )
        except Exception as e:
            outs = [f"[ERROR] {type(e).__name__}: {e}"] * RUNS_PER_PERTURB

        for run_id, out in enumerate(outs, start=1):
            job += 1
            f.write(f"run{run_id}- {out}\n")
        f.write("-" * 120 + "\n")
        f.flush()
        os.fsync(f.fileno())

        chars = sum(len(o) for o in outs) // max(1, len(outs))
        print(
            f"    {RUNS_PER_PERTURB} runs in {time.time() - tp:.1f}s"
            f" | global {job}/{total} | mean chars={chars}"
        )
        checkpoint_copy()

checkpoint_copy()
print(f"\nDone in {(time.time() - t0) / 60:.1f} min | log: {log_path}")
print(f"Local copy: {local_log}")